# DV-01/DV-02/DV-04: Pool141 Refresh, Prune & Inject Own-Track
**Goal:** Refresh stale pool141 (2 weeks old), prune duplicates, inject own-track ensemble
**Compute:** Kaggle Kernel (CPU only, ~10 min)
**Output:** 5 submission files for LB probing

---

In [ ]:
!pip install -q scikit-learn pandas numpy scipy

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import rankdata, spearmanr
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e8')
SUB_DIR = Path('/kaggle/working')
POOL_DIR = Path('/kaggle/input')  # Will search for pool submissions

def prog(msg):
    print(f"[POOL] {msg}", flush=True)

In [ ]:
# ─── Load own-track predictions ───
# First, we need the fixed own-track ensemble
# Check if it exists in working dir
own_track_paths = [
    SUB_DIR / 'submission_ensemble_fixed_0.96425.csv',
    SUB_DIR / 'submission_ensemble_0.96425.csv',
    SUB_DIR / 'submission_xgb_opt_fixed.csv',
    SUB_DIR / 'submission_lgb_fixed.csv',
]

own_track = None
for p in own_track_paths:
    if p.exists():
        own_track = pd.read_csv(p)['addicted_label'].values
        prog(f"Loaded own-track from {p}")
        break

if own_track is None:
    # Try to load from a known kernel output
    # You may need to add the dataset: adi/s6e8-own-track-ensemble
    raise FileNotFoundError("Own-track ensemble not found. Run final_submission_fixed kernel first.")

prog(f"Own-track: mean={own_track.mean():.5f}, std={own_track.std():.5f}, len={len(own_track)}")

In [ ]:
# ─── Collect public submissions from pool datasets ───
# Search all available datasets for submission CSVs
import os

pool_subs = {}

# Check common pool dataset locations
pool_locations = [
    Path('/kaggle/input'),  # All mounted datasets
    Path('/kaggle/input/playground-series-s6e8'),
]

for loc in pool_locations:
    if loc.exists():
        for root, dirs, files in os.walk(loc):
            for f in files:
                if f.endswith('.csv'):
                    fp = Path(root) / f
                    try:
                        df = pd.read_csv(fp)
                        if 'addicted_label' in df.columns and len(df) == 296302:
                            name = f"{fp.parent.name}/{fp.stem}"
                            if name not in pool_subs:
                                pool_subs[name] = df['addicted_label'].values
                                prog(f"  Loaded {name}: {len(df)} rows")
                    except Exception as e:
                        pass

prog(f"Total submissions collected: {len(pool_subs)}")

if len(pool_subs) < 2:
    prog("WARNING: Few submissions found. You may need to add pool datasets manually.")
    prog("Add datasets like: szymonkapiski/s6e8-oof-library-47-models, najiama/predicting-smartphone-addiction-oof-submission-csv, etc.")

In [ ]:
# ─── Pairwise rank correlations ───
def rank_transform(preds):
    return rankdata(preds) / len(preds)

names = list(pool_subs.keys())
n = len(names)
prog(f"Computing pairwise correlations for {n} submissions...")

ranked = {name: rank_transform(preds) for name, preds in pool_subs.items()}
corr_matrix = np.ones((n, n))

for i, name_i in enumerate(names):
    for j, name_j in enumerate(names):
        if i < j:
            rho, _ = spearmanr(ranked[name_i], ranked[name_j])
            corr_matrix[i, j] = corr_matrix[j, i] = rho

# Show top correlations
prog("Top pairwise correlations:")
for i in range(n):
    for j in range(i+1, n):
        if corr_matrix[i, j] > 0.99:
            prog(f"  {names[i]} <-> {names[j]}: {corr_matrix[i, j]:.4f}")

In [ ]:
# ─── Cluster at 0.999 threshold ───
threshold = 0.999
visited = [False] * n
clusters = []

for i in range(n):
    if visited[i]:
        continue
    cluster = [i]
    visited[i] = True
    for j in range(i+1, n):
        if not visited[j] and corr_matrix[i, j] >= threshold:
            cluster.append(j)
            visited[j] = True
    clusters.append(cluster)

prog(f"Found {len(clusters)} clusters at threshold {threshold}")

for idx, cluster in enumerate(clusters):
    cluster_names = [names[i] for i in cluster]
    prog(f"  Cluster {idx+1} ({len(cluster)}): {cluster_names[:5]}{'...' if len(cluster) > 5 else ''}")

In [ ]:
# ─── Prune: keep most diverse per cluster ───
keep_indices = []

for cluster in clusters:
    if len(cluster) == 1:
        keep_indices.append(cluster[0])
    else:
        best_idx = cluster[0]
        best_score = float('inf')
        for idx in cluster:
            avg_corr = np.mean([corr_matrix[idx, j] for j in cluster if j != idx])
            if avg_corr < best_score:
                best_score = avg_corr
                best_idx = idx
        keep_indices.append(best_idx)
        prog(f"  Kept {names[best_idx]} (avg corr to cluster: {best_score:.4f})")

pruned_names = [names[i] for i in keep_indices]
prog(f"Pruned from {len(names)} to {len(pruned_names)} submissions")

In [ ]:
# ─── Uniform rank average of pruned pool ───
pool_blend = np.zeros(296302)
for name in pruned_names:
    pool_blend += rank_transform(pool_subs[name])
pool_blend /= len(pruned_names)

prog(f"Pruned pool uniform rank-avg: mean={pool_blend.mean():.5f}")

In [ ]:
# ─── Inject own-track at different weights ───
injection_weights = [0.01, 0.02, 0.05, 0.10]
injected = {}

own_rank = rank_transform(own_track)
pool_rank = pool_blend  # already ranked

for w in injection_weights:
    blended = (1 - w) * pool_rank + w * own_rank
    injected[f'w_{int(w*100)}pct'] = blended
    prog(f"  {int(w*100)}% injection: mean={blended.mean():.5f}")

In [ ]:
# ─── Generate submission files ───
sub_ids = pd.read_csv(DATA_DIR / 'sample_submission.csv')['id']

# Base pool blend
base_path = SUB_DIR / 'submission_dv01_pruned_pool.csv'
pd.DataFrame({'id': sub_ids, 'addicted_label': np.clip(pool_blend, 0, 1)}).to_csv(base_path, index=False)
prog(f"Base pool: {base_path}")

# Injected variants
for name, preds in injected.items():
    path = SUB_DIR / f'submission_dv04_inject_{name}.csv'
    pd.DataFrame({'id': sub_ids, 'addicted_label': np.clip(preds, 0, 1)}).to_csv(path, index=False)
    prog(f"  {name}: {path}")

# Also save own-track alone for reference
own_path = SUB_DIR / 'submission_own_track_ref.csv'
pd.DataFrame({'id': sub_ids, 'addicted_label': np.clip(own_track, 0, 1)}).to_csv(own_path, index=False)
prog(f"Own-track reference: {own_path}")

In [ ]:
# ─── Download all submissions ───
from google.colab import files

files.download(str(base_path))
for name in injected:
    files.download(str(SUB_DIR / f'submission_dv04_inject_{name}.csv'))
files.download(str(own_path))

prog("\n=== SUBMISSIONS READY FOR LB PROBING ===")
prog("1. submission_dv01_pruned_pool.csv - baseline refreshed pool")
prog("2. submission_dv04_inject_w_1pct.csv - 1% own-track")
prog("3. submission_dv04_inject_w_2pct.csv - 2% own-track")
prog("4. submission_dv04_inject_w_5pct.csv - 5% own-track")
prog("5. submission_dv04_inject_w_10pct.csv - 10% own-track")
prog("\nSUGGESTED: Submit #1, #2, #3 to public LB (DV-13 showed 1% XGB best, higher = worse)")